### **Extraer el ` nombre del autor`, `fecha de nacimiento` y `biografia` y exportarlos a un archivo `JSON` de todas las páginas del sitioweb**

Vamos a trabajar en el enlace:

https://quotes.toscrape.com/

#### **Crear la spider**

Dado que en el archivo **`1-quotes_to_scrape.ipynb`** realizamos todo el proceso de creación desde el Entorno virtual, instalación de Scrapy y creación del proyecto, ahora solo nos queda comenzar a crear nuestras spiders. Para ello desde la terminal ejecutamos lo siguiente:

<center><img src="https://i.postimg.cc/25BC7TST/ws-396.png"></center>

Luego, debemos editar el archivo `quotes_to_scrape_siete.py` que se crea, con el código que se muestra abajo.

#### **Paso a paso**

Vamos a inspeccionar el sitio:

<center><img src="https://i.postimg.cc/NGP2bPqQ/ws-358.png"></center>

Localizar todos los tag `<a>` que vienen despues de un tag finalizado </> con `class="author"`

<center><img src="https://i.postimg.cc/J4szCHS9/ws-398.png"></center>

Ya desde la página del autor, localizar el `nombre` del autor:

<center><img src="https://i.postimg.cc/4xt4mkfP/ws-399.png"></center>

Localizar la `fecha de nacimiento` del autor:

<center><img src="https://i.postimg.cc/Lsp9PChp/ws-400.png"></center>

Localizar la `biografía` del autor:

<center><img src="https://i.postimg.cc/zfFJFCRR/ws-401.png"></center>

Desde la página principal, localizar si existe un botón de `Siguiente` para pasar a la siguiente página a continuar el proceso:

<center><img src="https://i.postimg.cc/zfjVhxFZ/ws-363.png"></center>

#### **Código**

Esta spider partirá de la página principal, seguirá todos los enlaces a las páginas de los autores llamando al callback `parse_author` para cada uno de ellos, y también los enlaces de paginación con el callback parse como vimos antes.

Aquí estamos pasando callbacks a `response.follow_all` como argumentos posicionales para hacer el código más corto; también funciona para `Request`.

El callback `parse_author` define una función de ayuda para extraer y limpiar los datos de una consulta `XPATH` y devuelve el dict Python con los datos de autor.

#########################################################

**Otra cosa interesante que demuestra esta spider es que, aunque haya muchas citas del mismo autor, no tenemos que preocuparnos por visitar varias veces la misma página de autor. `Por defecto, Scrapy filtra las peticiones duplicadas a URLs ya visitadas`, evitando el problema de golpear demasiado a los servidores por un error de programación. Esto puede configurarse mediante el parámetro `DUPEFILTER_CLASS`.**

Esperemos que a estas alturas tengas una buena comprensión de cómo utilizar el mecanismo de seguimiento de enlaces y callbacks con Scrapy.

#########################################################

**`RECORDAR`**: Para los elementos `<a>` existe un atajo: `response.follow` utiliza su atributo `href` automáticamente. Así se puede acortar aún más el código. Para crear `múltiples requests` a partir de un iterable, puede utilizar `response.follow_all` en su lugar.

In [ ]:
import scrapy


class QuotesToScrapeSieteSpider(scrapy.Spider):
    name = "quotes_to_scrape_siete"
    allowed_domains = ["quotes.toscrape.com"]
    start_urls = ["https://quotes.toscrape.com"]

    # def parse(self, response):
    #     author_page_links = response.css(".author + a")
    #     yield from response.follow_all(author_page_links, self.parse_author)

    #     pagination_links = response.css("li.next a")
    #     yield from response.follow_all(pagination_links, self.parse)

    def parse(self, response):
        # Toma todos los tag <a> que vienen despues de cerrado un tag con class="author"
        # No hay forma de lograr el mismo resultado con XPATH en este HTML
        # Es decir, va ir captando el 'enlace relativo' --> /author/Albert-Einstein/ 
        # y luego, iterando cada uno de ellos, lo va uniendo al enlace del request --> https://quotes.toscrape.com/author/Albert-Einstein/ 
        # Generando el response
        # Y utiliza la funcion 'parse_author' como callback para extraer los datos de cada uno del enlace resultante
        author_page_links = response.css(".author + a")
        yield from response.follow_all(author_page_links, self.parse_author)

        # Después ..
        # Va a iterar sobre la variable 'pagination_links' y obtendra todos los 'Enlaces relativos', en el boton, de 'Pagina siguiente' 
        # es decir, obtenemos '/page/2/', '/page/3/' '/page/4/', etc... Hasta que ya no existan más
        # Esto es lo que nos permite el metodo '.follow_all()'
        pagination_links = response.xpath('//li[@class="next"]/a')
        yield from response.follow_all(pagination_links, self.parse)

    # def parse_author(self, response):
    #     def extract_with_css(query):
    #         return response.css(query).get(default="").strip()

    #     yield {
    #         "name": extract_with_css("h3.author-title::text"),
    #         "birthdate": extract_with_css(".author-born-date::text"),
    #         "bio": extract_with_css(".author-description::text"),
    #     }

    def parse_author(self, response):

        def extract_with_xpath(query):
            # get(default="") --> Si no encuentra ningún valor devolverá un valor vacío
            return response.xpath(query).get(default="").strip()

        yield {
            "name": extract_with_xpath('//h3[@class="author-title"]/text()'),
            "birthdate": extract_with_xpath('//span[@class="author-born-date"]/text()'),
            "bio": extract_with_xpath('//div[@class="author-description"]/text()'),
        }

#### **Ejecución**

Nos posicionamos en la ruta correcta y ejecutamos nuestra Spider:

In [ ]:
scrapy crawl quotes_to_scrape_siete

<center><img src="https://i.postimg.cc/4yYy0T70/ws-402.png"></center>

De esta manera obtenemos toda la info que pedimos extraer del website:

<center><img src="https://i.postimg.cc/054bwrh1/ws-403.png"></center>

Podriamos exportar toda esta data en un archivo `JSON` escribiendo el siguiente comando:

In [ ]:
scrapy crawl quotes_to_scrape_siete -o quotes_to_scrape_siete.json

<center><img src="https://i.postimg.cc/yx2x4r08/ws-404.png"></center>